In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", 
            "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", 
            "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+--------------------+-------------------+--------------------+--------------------+-----+
|                 _id|           categoria|      fecha_captura|               grupo|                item|valor|
+--------------------+--------------------+-------------------+--------------------+--------------------+-----+
|69d65f6d075fae0d6...|              Travel|2026-04-08 14:00:13|        G1_Ecommerce|  It’s Only Himalaya|45.17|
|69d65f6d075fae0d6...|              Travel|2026-04-08 14:00:13|        G1_Ecommerce|Full Moon over N ...|49.43|
|69d66210075fae0d6...|Turismo y Hosteleria|2026-04-08 14:11:28|G5_Turismo_y_Host...|Hotel Patagonia E...|45.17|
|69d66210075fae0d6...|Turismo y Hosteleria|2026-04-08 14:11:28|G5_Turismo_y_Host...|Hostal 

In [5]:
# Esto nos dice que columnas entendio Spark que habia en Mongo
df.printSchema()

# Esto nos muestra las primeras 5 filas para confirmar que hay datosreales
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+--------------------+-------------------+--------------------+--------------------+-----+
|                 _id|           categoria|      fecha_captura|               grupo|                item|valor|
+--------------------+--------------------+-------------------+--------------------+--------------------+-----+
|69d65f6d075fae0d6...|              Travel|2026-04-08 14:00:13|        G1_Ecommerce|  It’s Only Himalaya|45.17|
|69d65f6d075fae0d6...|              Travel|2026-04-08 14:00:13|        G1_Ecommerce|Full Moon over N ...|49.43|
|69d66210075fae0d6...|Turismo y Hosteleria|2026-04-08 14:11:28|G5_Turismo_y_Host...|Hotel Patagonia E...|45.17|
|69d66210075fae0d6...|Turismo y Hosteleria|2026-04-08 14:11:28|G5_Turismo_y_Host...|Hostal 

In [6]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 6
Registros validos: 6


In [7]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|Hotel Patagonia E...|45.17|
|Hostal Desierto d...|49.43|
|  It's Only Himalaya|45.17|
|Full Moon over No...|49.43|
+--------------------+-----+



In [9]:
df.filter(df["valor"] > 100).show()

+---+---------+-------------+-----+----+-----+
|_id|categoria|fecha_captura|grupo|item|valor|
+---+---------+-------------+-----+----+-----+
+---+---------+-------------+-----+----+-----+



In [10]:
df.groupBy("grupo").count().show()

+--------------------+-----+
|               grupo|count|
+--------------------+-----+
|G5_Turismo_y_Host...|    4|
|        G1_Ecommerce|    2|
+--------------------+-----+



In [11]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+--------------------+---------------+---------------+-------------+-------------+
|           categoria|cantidad_libros|precio_promedio|precio_minimo|precio_maximo|
+--------------------+---------------+---------------+-------------+-------------+
|              Travel|              4|           47.3|        45.17|        49.43|
|Turismo y Hosteleria|              2|           47.3|        45.17|        49.43|
+--------------------+---------------+---------------+-------------+-------------+



In [12]:
from pyspark.sql import functions as F
# 1. Ordenar por valor de forma descendente
# 2. Tomar el primer registro
ganador = df.orderBy(F.desc("valor")).limit(1)

ganador.select("item", "valor", "grupo", "fecha_captura").show()

+--------------------+-----+------------+-------------------+
|                item|valor|       grupo|      fecha_captura|
+--------------------+-----+------------+-------------------+
|Full Moon over N ...|49.43|G1_Ecommerce|2026-04-08 14:00:13|
+--------------------+-----+------------+-------------------+



In [ ]:
from pyspark.sql import functions as F

NOMBRE_GRUPO = "G5_Turismo_y_Hosteleria"

ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()